In [0]:
# Read the raw sellers CSV using Auto Loader
# with schema hints for non-string types, schema inference, and schema evolution enabled

df_sellers_bronze = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/second_data_engineering_project/pipeline_metadata/autoloader_metadata/schemas/sellers") \
    .option("cloudFiles.inferColumnTypes", "true") \
    .option("cloudFiles.schemaHints", "seller_zip_code_prefix INT") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("rescuedDataColumn", "_rescued_data") \
    .load("/Volumes/second_data_engineering_project/landing/raw_files/olist_sellers_dataset")

In [0]:
# With Auto Loader streaming, we can inspect the inferred schema
# Data preview and counts are available after writing to the Delta table

df_sellers_bronze.printSchema()

In [0]:
# Write the streaming Bronze DataFrame as a Delta table
# Using Trigger.AvailableNow for batch-like processing with Auto Loader benefits
# mergeSchema allows the Delta table to accept new columns discovered by Auto Loader
# Checkpoint location enables incremental processing on subsequent runs

df_sellers_bronze.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("mergeSchema", "true") \
    .option("checkpointLocation", "/Volumes/second_data_engineering_project/pipeline_metadata/autoloader_metadata/checkpoints/bronze/sellers") \
    .trigger(availableNow=True) \
    .toTable("second_data_engineering_project.bronze.sellers")

In [0]:
%sql
-- View actual rescued data if any exists

SELECT *
FROM second_data_engineering_project.bronze.sellers
WHERE _rescued_data IS NOT NULL
LIMIT 100;

In [0]:
%sql
-- Validate the Auto Loader output by counting rows in the Bronze sellers table

SELECT COUNT(*) AS row_count
FROM second_data_engineering_project.bronze.sellers;

In [0]:
%sql
-- Display sample data from the Bronze sellers table

SELECT *
FROM second_data_engineering_project.bronze.sellers
LIMIT 100;